In [ ]:
! pip install mem0ai openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.5/330.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.6/394.6 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 24.6 MB/s eta 0:00:00


In [ ]:
"""
Segment 4 Exercise — Add Mem0 to an Existing Agent
====================================================

Goal: take a completely bare chatbot (zero memory) and add Mem0 to it.
When a user says "my name is X" or "I prefer Y" in session 1, the agent
should recall both of those in session 2 — without being re-told.

Setup:
    pip install mem0ai openai
    export MEM0_API_KEY="your-mem0-key"
    export OPENAI_API_KEY="your-openai-key"

If you're in Google Colab: toggling a Secret's notebook-access switch does
NOT automatically set it as an environment variable — the code below loads
it into os.environ for you. This has to happen again after every runtime
restart/disconnect, since that wipes os.environ. If you ever see
"OpenAIError: Missing credentials" again after things were working, this
is almost always why — just re-run the notebook from the top.

Note: current mem0ai SDK versions require user_id to be passed inside a
filters dict on search() rather than as a top-level kwarg — e.g.
mem0.search(query, filters={"user_id": user_id}) instead of
mem0.search(query, user_id=user_id). Some versions also wrap the result
list inside a dict, e.g. {"results": [...]}, instead of returning a bare
list — the code below checks for and unwraps that shape automatically.
If add() throws a similar error in your installed version, apply the
same filters={"user_id": ...} pattern there too.
"""

import os

# --- Colab only: pull secrets into environment variables ---
# Every fresh Colab runtime (including after a restart or disconnect)
# wipes os.environ, so this has to run again each time before OpenAI()
# or MemoryClient() are created. Safe to leave in outside Colab too —
# it just does nothing if google.colab isn't available.
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    os.environ["MEM0_API_KEY"] = userdata.get("MEM0_API_KEY")
except ImportError:
    pass  # not running in Colab — assume env vars are already set

from openai import OpenAI
from mem0 import MemoryClient

llm  = OpenAI()
mem0 = MemoryClient(api_key=os.getenv("MEM0_API_KEY"))

SYSTEM = "You are a helpful assistant."


def chat(user_msg, user_id, debug=False):
    # 1. Retrieve relevant memories — semantic search, scoped to this user
    results = mem0.search(user_msg, filters={"user_id": user_id})
    # Newer mem0ai versions wrap results as {"results": [...]} rather than
    # returning a bare list — handle both shapes defensively.
    if isinstance(results, dict):
        results = results.get("results", [])
    if debug:
        print(f"[debug] search returned {len(results)} memories: {results}")
    ctx = "\n".join(m["memory"] for m in results)

    messages = [
        {"role": "system", "content": f"{SYSTEM}\n\nWhat you know about this user:\n{ctx}"},
        {"role": "user", "content": user_msg},
    ]

    resp = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
    )
    reply = resp.choices[0].message.content

    # 2. Store new facts automatically — Mem0 extracts what's worth keeping.
    # Pass this as a conversation turn (list of message dicts), not a raw
    # string — that's what reliably triggers Mem0's LLM-based fact
    # extraction pipeline in current SDK versions.
    add_result = mem0.add(
        [{"role": "user", "content": user_msg}],
        user_id=user_id,
    )
    if debug:
        print(f"[debug] add() stored: {add_result}")

    return reply


if __name__ == "__main__":
    user_id = "u42"

    # ---- Session 1: tell the agent about yourself ----
    print("--- Session 1 ---")
    print(chat("My name is Alex.", user_id, debug=True))
    print(chat("I prefer concise, bullet-point answers.", user_id, debug=True))

    # Optional: confirm memories actually landed before session 2 runs.
    # If this prints an empty list, add() isn't persisting — check the
    # debug output above and Mem0's dashboard for this user_id.
    import time
    time.sleep(2)  # small buffer — Mem0's extraction can be async
    stored = mem0.get_all(filters={"user_id": user_id})
    if isinstance(stored, dict):
        stored = stored.get("results", [])
    print(f"\n[debug] all memories currently stored for {user_id}: {stored}\n")

    # ---- Session 2: fresh conversation, no context carried over manually ----
    print("--- Session 2 (brand new session) ---")
    print(chat("What's my name, and how do I like my answers formatted?", user_id, debug=True))

    # Expected: the agent recalls "Alex" and "concise, bullet-point answers"
    # even though session 2 never mentioned either — Mem0 retrieved them
    # from what session 1 stored.

--- Session 1 ---


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[debug] search returned 2 memories: [{'id': 'e6b58946-7831-4137-8a77-f7b27d2d0fe5', 'memory': "User's name is Alex.", 'user_id': 'u42', 'agent_id': None, 'app_id': None, 'run_id': None, 'score': 0.3665, 'score_breakdown': {'semantic': 0.8635, 'bm25': 0.0526, 'entity': 0.0}, 'metadata': {}, 'categories': ['personal_details'], 'created_at': '2026-07-06T10:56:34+00:00', 'updated_at': '2026-07-06T10:56:41.701966+00:00', 'expiration_date': None}, {'id': 'd807563c-8785-4d1a-830f-0a708ddbb3d0', 'memory': 'User prefers concise, bullet-point answers', 'user_id': 'u42', 'agent_id': None, 'app_id': None, 'run_id': None, 'score': 0.1322, 'score_breakdown': {'semantic': 0.3306, 'bm25': 0.0, 'entity': 0.0}, 'metadata': {}, 'categories': ['user_preferences'], 'created_at': '2026-07-06T10:56:36+00:00', 'updated_at': '2026-07-06T10:56:42.874040+00:00', 'expiration_date': None}]
[debug] add() stored: {'event_id': 'c4158e39-1430-44de-8175-457b9ce25998', 'status': 'PENDING'}
Hello, Alex! How can I assist 